> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 8 · Notebook 05 — Parameter search, stability and trade-offs

**Sessions:** S9 (Search spaces, objectives & stability) · S10 (Bayesian, genetic & multi-objective optimization) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Run a grid search and log every trial.
2. Score parameters by the plateau around them, not by their peak.
3. Find the Pareto front between Sharpe and turnover.
4. Compare grid and random search on the same budget.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. The objective

Time-series momentum (Part 7) with two parameters: the momentum `lookback` and the volatility window `vol_n`. Optimize on the **first half** of the known-regime market only; keep the second half for later.

In [ ]:
bars = p.regime_market()
o, c = bars.open.to_numpy(), bars.close.to_numpy()
half = len(c) // 2

def pnl(lookback, vol_n):
    return p.first_look_pnl(p.tsmom_signal(c, lookback, vol_n), o)

def in_sample_sharpe(lookback, vol_n):
    return p.sharpe(pnl(lookback, vol_n)[:half])

grid = {"lookback": [20, 40, 60, 90, 120, 160, 200, 250], "vol_n": [10, 20, 40, 60, 90, 120]}

## 2. Grid search

Evaluate the objective on **every combination** (`itertools.product` over the grid's values, in the grid's key order) and return a DataFrame with one column per parameter plus `score`, one row per combination in evaluation order.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import itertools

def grid_search(objective, grid):
    keys = list(grid)
    rows = []
    for combo in itertools.product(*grid.values()):
        params = dict(zip(keys, combo))
        rows.append(params | {"score": objective(**params)})
    return pd.DataFrame(rows)

mine = p.attempt(grid_search, in_sample_sharpe, grid)
mine = p.check("grid_search", mine, p.grid_search(in_sample_sharpe, grid))
mine.pivot(index="vol_n", columns="lookback", values="score").round(2)

48 trials. The best in-sample score is the most optimistic number in the table: it is the maximum of 48 noisy estimates.

## 3. Peaks and plateaus

A parameter set on an isolated spike is probably fitted to noise; one in the middle of a broad high region survives small changes. Replace each cell of the `(vol_n × lookback)` table by the **mean of its neighbourhood**: the cells within `radius` grid steps in both directions, clipped at the edges (`np.nanmean`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def plateau_scores(results, x, y, radius=1):
    tab = results.pivot(index=y, columns=x, values="score").sort_index().sort_index(axis=1)
    a = tab.to_numpy()
    out = np.empty_like(a)
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            out[i, j] = np.nanmean(a[max(0, i - radius):i + radius + 1, max(0, j - radius):j + radius + 1])
    return pd.DataFrame(out, index=tab.index, columns=tab.columns)

res = p.grid_search(in_sample_sharpe, grid)
mine = p.attempt(plateau_scores, res, "lookback", "vol_n")
mine = p.check("plateau_scores", mine, p.plateau_scores(res, "lookback", "vol_n"))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (t, tab) in zip(axes, [("raw in-sample Sharpe", res.pivot(index="vol_n", columns="lookback", values="score")), ("plateau score", mine)]):
    im = ax.imshow(tab.to_numpy(), cmap="viridis", aspect="auto"); plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(tab.columns)), tab.columns); ax.set_yticks(range(len(tab.index)), tab.index)
    ax.set(xlabel="lookback", ylabel="vol_n", title=t)
plt.tight_layout(); plt.show()

In [ ]:
pl = p.plateau_scores(res, "lookback", "vol_n")
peak = res.loc[res.score.idxmax()]
i, j = np.unravel_index(np.nanargmax(pl.to_numpy()), pl.shape)
choices = {"peak": (int(peak.lookback), int(peak.vol_n)), "plateau": (int(pl.columns[j]), int(pl.index[i]))}
oos = {(r.lookback, r.vol_n): p.sharpe(pnl(int(r.lookback), int(r.vol_n))[half:]) for r in res.itertuples()}
for k, (lb, vn) in choices.items():
    print(f"{k:8s} lookback {lb:3d}, vol_n {vn:3d}: in-sample {in_sample_sharpe(lb, vn):.2f}, out-of-sample {oos[(lb, vn)]:.2f}")
print(f"correlation of in-sample and out-of-sample Sharpe across the 48 trials: {np.corrcoef(res.score, list(oos.values()))[0, 1]:.2f}")

Here the best lookback (160) is a whole **ridge**, so the peak and the plateau choice agree and both hold up out of sample. That is the good case. When the peak is an isolated spike, the plateau score moves you to safer ground; it costs nothing when it isn't needed.

## 4. More than one objective

Sharpe isn't the only thing that matters: turnover costs money and capacity (notebook 04). A configuration is on the **Pareto front** if no other configuration is at least as good on both criteria and strictly better on one. Return the non-dominated rows sorted by turnover.

In [ ]:
rows = []
for lb in grid["lookback"]:
    for vn in grid["vol_n"]:
        sig = p.tsmom_signal(c, lb, vn)
        rows.append({"lookback": lb, "vol_n": vn, "sharpe": p.sharpe(p.first_look_pnl(sig, o)),
                     "turnover": np.abs(np.diff(sig)).sum() / len(sig) * 252})
trials = pd.DataFrame(rows)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pareto_front(df, maximize, minimize):
    a, b = df[maximize].to_numpy(), df[minimize].to_numpy()
    keep = []
    for i in range(len(df)):
        dominated = np.any((a >= a[i]) & (b <= b[i]) & ((a > a[i]) | (b < b[i])))
        if not dominated:
            keep.append(i)
    return df.iloc[keep].sort_values(minimize)

mine = p.attempt(pareto_front, trials, "sharpe", "turnover")
mine = p.check("pareto_front", mine, p.pareto_front(trials, "sharpe", "turnover"))
plt.scatter(trials.turnover, trials.sharpe, s=15, color="#b5b4ad", label="all 48 trials")
plt.plot(mine.turnover, mine.sharpe, "o-", color=p.PALETTE[1], label="Pareto front")
plt.xlabel("turnover (× capital per year)"); plt.ylabel("Sharpe"); plt.legend(); plt.show()
mine.round(3)

## 5. Grid vs random search

With the same budget of 48 trials, random search tries 48 *different* values of each parameter instead of 8 and 6. When only one or two parameters matter, that finds better regions (Bergstra & Bengio, 2012). Bayesian optimizers (Optuna, used in the lab) go further by learning where to look next. All of them make the best score **more** optimistic, which is why every trial is logged.

In [ ]:
space = {"lookback": list(range(20, 251, 10)), "vol_n": list(range(10, 121, 5))}
rnd = p.random_search(in_sample_sharpe, space, n=48, seed=0)
print(f"grid:   best in-sample {res.score.max():.2f} from {res.lookback.nunique()}×{res.vol_n.nunique()} distinct values")
print(f"random: best in-sample {rnd.score.max():.2f} from {rnd.lookback.nunique()}×{rnd.vol_n.nunique()} distinct values")

## Wrap-up

* Declare the search space, log every trial, and score stability (plateaus), not peaks.
* Trade-offs are a front, not a single number.
* Graded version: `labs/part08/week27_optimization` (grid, random and Optuna search, plateau scores).